In [ ]:
# Install FAISS and the document-reading libraries
!pip install -q faiss-cpu scikit-learn PyPDF2 python-docx fpdf2 ollama

# Install zstd, which is required by ollama for extraction
!apt-get install -y zstd

!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

subprocess.Popen(["ollama","serve"])

time.sleep(5)

!ollama pull qwen2.5:1.5b

# Import necessary libraries
import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import faiss

import PyPDF2
from docx import Document
from fpdf import FPDF
import ollama
print("Libraries installed and imported.")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

Libraries installed and imported.


In [ ]:
# ---- Step 1: Create sample PDF and Word files (stand-ins for real documents) ----

course_descriptions = {
    'CS101': ('Introduction to Computer Science',
              'Covers fundamental concepts of programming, algorithms, and data structures. Introduces Python.'),
    'MA201': ('Calculus I',
              'First course in a sequence of calculus, covering limits, derivatives, and integrals.'),
    'PH101': ('General Physics',
              'An introductory course to the principles of classical mechanics and thermodynamics.'),
    'BI301': ('Molecular Biology',
              'Explores the structure and function of biological macromolecules, genes, and cells.'),
    'HI105': ('World History I',
              'A survey of major global historical events from ancient civilizations to the early modern period.'),
}

# Three courses go into a PDF, one course per page
pdf = FPDF()
for course_id in ['CS101', 'MA201', 'PH101']:
    title, description = course_descriptions[course_id]
    pdf.add_page()
    pdf.set_font("Helvetica", size=12)
    pdf.multi_cell(0, 8, f"{course_id} - {title}: {description}")
pdf.output("course_catalog.pdf")

# Two courses go into a Word document, one course per paragraph
doc = Document()
for course_id in ['BI301', 'HI105']:
    title, description = course_descriptions[course_id]
    doc.add_paragraph(f"{course_id} - {title}: {description}")
doc.save("course_catalog.docx")

print("Created course_catalog.pdf and course_catalog.docx")


# ---- Step 2: Extract text from each file, one chunk per page (PDF) / paragraph (Word) ----

def extract_pdf_chunks(path):
    """Read a PDF and return one chunk per page."""
    reader = PyPDF2.PdfReader(path)
    chunks = []
    for i, page in enumerate(reader.pages):
        text = (page.extract_text() or "").replace("\n", " ").strip()
        if text:
            chunks.append({"source_file": os.path.basename(path), "file_type": "PDF", "chunk_id": i, "text": text})
    return chunks


def extract_docx_chunks(path):
    """Read a .docx file and return one chunk per non-empty paragraph."""
    doc = Document(path)
    paragraphs = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    return [
        {"source_file": os.path.basename(path), "file_type": "DOCX", "chunk_id": i, "text": p}
        for i, p in enumerate(paragraphs)
    ]


chunks = extract_pdf_chunks("course_catalog.pdf") + extract_docx_chunks("course_catalog.docx")

# Create a Pandas DataFrame -- our new knowledge base, sourced from real files
df = pd.DataFrame(chunks)

print(f"\nExtracted {len(df)} text chunks from 1 PDF and 1 DOCX file:")
display(df)

Created course_catalog.pdf and course_catalog.docx

Extracted 5 text chunks from 1 PDF and 1 DOCX file:


,source_file,file_type,chunk_id,text
0,course_catalog.pdf,PDF,0,CS101 - Introduction to Computer Science: Cove...
1,course_catalog.pdf,PDF,1,MA201 - Calculus I: First course in a sequence...
2,course_catalog.pdf,PDF,2,PH101 - General Physics: An introductory cours...
3,course_catalog.docx,DOCX,0,BI301 - Molecular Biology: Explores the struct...
4,course_catalog.docx,DOCX,1,HI105 - World History I: A survey of major glo...


In [ ]:
# Initialize the TF-IDF Vectorizer
# It converts a collection of raw documents to a matrix of TF-IDF features
tfidf_vectorizer = TfidfVectorizer()

# Fit the vectorizer to our extracted text chunks and transform them into TF-IDF vectors
# .fit_transform learns the vocabulary and IDF from the training data and transforms it
tfidf_matrix = tfidf_vectorizer.fit_transform(df['text'])

# Convert the sparse matrix to a dense NumPy array, which FAISS requires
document_embeddings = tfidf_matrix.toarray().astype('float32')

# Print the shape of the TF-IDF matrix to see its dimensions
print(f"Shape of TF-IDF embeddings (chunks x features): {document_embeddings.shape}")

Shape of TF-IDF embeddings (chunks x features): (5, 60)


In [ ]:
# Get the dimensionality of our embeddings
d = document_embeddings.shape[1]

# Initialize a FAISS index
# IndexFlatL2 creates a simple index that performs exhaustive search using L2 (Euclidean) distance
index = faiss.IndexFlatL2(d)

# Add the document embeddings to the FAISS index
# This builds the searchable database
index.add(document_embeddings)

print(f"FAISS index created with {index.ntotal} documents.")
print(f"Index is trained: {index.is_trained}")

FAISS index created with 5 documents.
Index is trained: True


In [ ]:
# Define a sample user question
user_question = "What is covered in the introductory programming course?"
print(f"User question: '{user_question}'")

# Transform the user question into a TF-IDF vector
# We use .transform() here because the vectorizer is already fitted with our corpus vocabulary
question_vector = tfidf_vectorizer.transform([user_question]).toarray().astype('float32')

print(f"Shape of question vector: {question_vector.shape}")

User question: 'What is covered in the introductory programming course?'
Shape of question vector: (1, 60)


In [ ]:
# Perform a similarity search using the FAISS index
# We want to retrieve the top 1 most similar document (k=1)
# D contains distances, I contains the indices of the retrieved documents
k = 1
distances, indices = index.search(question_vector, k)

# Get the index of the most similar document
most_similar_doc_index = indices[0][0]

# Retrieve the actual document content using its index
retrieved_document = df.iloc[most_similar_doc_index]

print("--- Retrieved Document (Context) ---")
display(retrieved_document)

--- Retrieved Document (Context) ---


,2
source_file,course_catalog.pdf
file_type,PDF
chunk_id,2
text,PH101 - General Physics: An introductory cours...


In [ ]:
# Extract the text and source file from the retrieved chunk
# Extract retrieved context

retrieved_text = retrieved_document["text"]
retrieved_source = retrieved_document["source_file"]

MODEL = "qwen2.5:1.5b"

prompt = f"""
You are a professional AI university assistant.

Answer the question ONLY using the provided context.

Question:
{user_question}

Context:
{retrieved_text}

Instructions:
1. Give a detailed answer.
2. Use simple English.
3. Do not make up information.
4. Explain properly.
"""

response = ollama.chat(
    model=MODEL,
    messages=[
        {
            "role":"user",
            "content":prompt
        }
    ]
)

print("="*60)
print("USER QUESTION")
print("="*60)
print(user_question)

print()

print("="*60)
print("RETRIEVED DOCUMENT")
print("="*60)
print(retrieved_source)

print()

print("="*60)
print("LLM GENERATED ANSWER")
print("="*60)
print(response["message"]["content"])

USER QUESTION
What is covered in the introductory programming course?

RETRIEVED DOCUMENT
course_catalog.pdf

LLM GENERATED ANSWER
The introductory programming course, PH101, primarily covers fundamental concepts in computer science and programming language basics. This course is designed to introduce students to the essential principles of software development, algorithm design, problem-solving techniques, and practical applications using a specific programming language or environment.

Key topics covered include:

1. Basic Programming Constructs:
   - Variables: Definition, declaration, types, initialization
   - Data Types (integers, floating-point numbers, strings)
   - Operators: Arithmetic, relational, logical, conditional statements

2. Control Structures:
   - Conditional Statements (if-else)
   - Loops (for, while, do-while)

3. Functions and Modules:
   - Defining functions
   - Modular programming practices
   - Passing arguments to functions

4. Basic Input/Output Operation